# Regime calssification

Using only information available today, can we predict which volatility regime the next five trading days will fall into ?

In [1]:
import pandas as pd

from market_risk.modelling.dataset import TEST_YEARS, load_model_dataset, create_walk_forward_fold, prepare_train_test
from market_risk.evaluation.regimes import get_regime_threshold, classify_regime

In [ ]:
# Load daatset
df = load_model_dataset()
df.head()

,date,sp500,vix,treasury_10y,treasury_2y,high_yield_spread,return_1d,return_5d,volatility_5d,volatility_10d,volatility_20d,vix_change_1d,vix_change_5d,vix_vs_realized_vol,yield_curve,treasury_10y_change_5d,treasury_2y_change_5d,target_volatility_5d
0,2016-08-08,2180.89,11.50,1.59,0.74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.85,NaN,NaN,0.099270
1,2016-08-09,2181.74,11.66,1.55,0.71,NaN,0.000390,NaN,NaN,NaN,NaN,0.16,NaN,NaN,0.84,NaN,NaN,0.131997
2,2016-08-10,2175.49,12.05,1.50,0.69,NaN,-0.002869,NaN,NaN,NaN,NaN,0.39,NaN,NaN,0.81,NaN,NaN,0.127387
3,2016-08-11,2185.79,11.68,1.57,0.76,NaN,0.004723,NaN,NaN,NaN,NaN,-0.37,NaN,NaN,0.81,NaN,NaN,0.108727
4,2016-08-12,2184.05,11.55,1.51,0.71,NaN,-0.000796,NaN,NaN,NaN,NaN,-0.13,NaN,NaN,0.80,NaN,NaN,0.110387


## Inspect the 2025 fold

We start with 2025 because we alreafy know it contains an interesting stress episode, as shown in `01_data_exploratory.ipynb`.

In [3]:
train, test = create_walk_forward_fold(df, 2025)
X_train, y_train, X_test, y_test = prepare_train_test(train, test)

# calculate the thresholds
elevated_threshold, stress_threshold = get_regime_threshold(y_train)
print(f"Elevated threshold: {elevated_threshold:.5f}")
print(f"Stress threshold: {stress_threshold:.5f}")

Elevated threshold: 0.35829
Stress threshold: 0.57310


In [20]:
train.head()

,date,sp500,vix,treasury_10y,treasury_2y,high_yield_spread,return_1d,return_5d,volatility_5d,volatility_10d,volatility_20d,vix_change_1d,vix_change_5d,vix_vs_realized_vol,yield_curve,treasury_10y_change_5d,treasury_2y_change_5d,target_volatility_5d
0,2016-08-08,2180.89,11.50,1.59,0.74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.85,NaN,NaN,0.099270
1,2016-08-09,2181.74,11.66,1.55,0.71,NaN,0.000390,NaN,NaN,NaN,NaN,0.16,NaN,NaN,0.84,NaN,NaN,0.131997
2,2016-08-10,2175.49,12.05,1.50,0.69,NaN,-0.002869,NaN,NaN,NaN,NaN,0.39,NaN,NaN,0.81,NaN,NaN,0.127387
3,2016-08-11,2185.79,11.68,1.57,0.76,NaN,0.004723,NaN,NaN,NaN,NaN,-0.37,NaN,NaN,0.81,NaN,NaN,0.108727
4,2016-08-12,2184.05,11.55,1.51,0.71,NaN,-0.000796,NaN,NaN,NaN,NaN,-0.13,NaN,NaN,0.80,NaN,NaN,0.110387


We get the same values as in `regimes.py`, that is a good consistency check.

### 1. Create training and test labels

In [46]:
y_train_class = pd.DataFrame(
    {
        "date": train.loc[y_train.index, "date"],
        "regime": y_train.apply(lambda x: classify_regime(x, elevated_threshold, stress_threshold)),
    }
)

y_test_class = pd.DataFrame(
    {
        "date": test.loc[y_test.index, "date"],
        "regime": y_test.apply(lambda x: classify_regime(x, elevated_threshold, stress_threshold)),
    }
)

y_train_class["date"] = pd.to_datetime(y_train_class["date"])
y_test_class["date"] = pd.to_datetime(y_test_class["date"])

Manual check

In [26]:
y_train_class.head()

,date,label
20,2016-09-06,Elevated
21,2016-09-07,Elevated
22,2016-09-08,Elevated
23,2016-09-09,Elevated
24,2016-09-12,Normal


In [27]:
y_test_class.head()

,date,label
2114,2025-01-02,Elevated
2115,2025-01-03,Normal
2116,2025-01-06,Normal
2117,2025-01-07,Elevated
2118,2025-01-08,Elevated


### 2. Check training proportions

In [47]:
y_train_class["regime"].value_counts(normalize=True)

regime
Normal      0.699856
Elevated    0.200096
Stress      0.100048
Name: proportion, dtype: float64

The training distribution roughly reflect : 
70% Normal
20% Elevated
10% Stress

because we initially defined the thresholds as :
70th percentile and 90th percentile

### 3. Inspect test distributions

In [48]:
y_test_class["regime"].value_counts(normalize=True)

regime
Normal      0.744
Elevated    0.188
Stress      0.068
Name: proportion, dtype: float64

Unlike the training distribution, we do not expect 70/20/10 here.

The thresholds came from historical training conditions. If 2025 was unusually volatile, for example, it could naturally contain proportionally more Elevated or Stress observations.

We must not adjust the thresholds to force the test set into a particular distribution.

### 4. Inspect labels alongside the actual target

In [42]:
regime_check = pd.DataFrame(
    {
        "date": y_test_class["date"],
        "target_volatility_5d": y_test,
        "regime": y_test_class["label"]
    }
)

regime_check["date"] = pd.to_datetime(regime_check["date"])
regime_check.sort_values(by="target_volatility_5d", ascending=False).head()

,date,target_volatility_5d,regime
2175,2025-04-02,1.929532,Stress
2176,2025-04-03,1.848132,Stress
2178,2025-04-07,1.598282,Stress
2177,2025-04-04,1.593768,Stress
2179,2025-04-08,1.578654,Stress


Check specifically April 2025

In [44]:
regime_check.loc[regime_check["date"].between("2025-04-01", "2025-04-30"), ["date", "regime"]]


,date,regime
2174,2025-04-01,Stress
2175,2025-04-02,Stress
2176,2025-04-03,Stress
2177,2025-04-04,Stress
2178,2025-04-07,Stress
2179,2025-04-08,Stress
2180,2025-04-09,Stress
2181,2025-04-10,Elevated
2182,2025-04-11,Elevated
2183,2025-04-14,Stress


We see the major forward-volatility observations around the April shock classified as Stress.

### Create the dummy classifier